# Convolutional Restricted Boltzmann Machine for MNIST
## A CNN Energy Function for the Boltzmann Machine

This notebook replaces the **flat weight matrix** $\boldsymbol{W}$ of
the standard Binary-Binary RBM with a **convolutional neural network**
as the energy function, then trains it on MNIST to generate new digits.

---

**Contents**

| Section | Topic |
|:--------|:------|
| 1 | Motivation: from flat RBM to ConvRBM |
| 2 | Imports and helpers |
| 3 | CNN Encoder and Decoder |
| 4 | ConvRBM class |
| 5 | MNIST data loading |
| 6 | Training |
| 7 | Generation — new digit images from Gibbs sampling |
| 8 | Reconstruction quality |
| 9 | Summary |

## 1. Motivation: from Flat RBM to ConvRBM

### Standard Binary-Binary RBM (recap)

The BB-RBM energy is (lecture notes cell 199):

$$E_{BB}(\boldsymbol{x},\boldsymbol{h};\boldsymbol{\Theta})
= -\boldsymbol{a}^T\boldsymbol{x}
  - \boldsymbol{b}^T\boldsymbol{h}
  - \boldsymbol{x}^T\boldsymbol{W}\boldsymbol{h}$$

For a $28\times28$ MNIST image flattened to $\boldsymbol{x}\in\{0,1\}^{784}$,
the weight matrix $\boldsymbol{W}\in\mathbb{R}^{784\times N}$ connects
**every pixel to every hidden unit** with no regard for spatial structure.
This is expensive and ignores the translational symmetry of digit images.

### Convolutional RBM

We keep the energy structure but replace the linear encoder
$\boldsymbol{x}\mapsto \boldsymbol{W}^T\boldsymbol{x}$
with a **CNN** $f_{\rm enc}$:

$$E_{\rm conv}(\boldsymbol{x},\boldsymbol{h})
= -\boldsymbol{a}^T\boldsymbol{x}
  - \boldsymbol{b}^T\boldsymbol{h}
  - \boldsymbol{h}^T f_{\rm enc}(\boldsymbol{x})$$

The CNN shares weights across spatial positions (via convolution), dramatically
reducing the parameter count and encoding local structure in the features.

### Free energy and probability (same structure as BB-RBM)

Marginalising over $\boldsymbol{h}$ (lecture notes cell 205):

$$F(\boldsymbol{x})
= -\boldsymbol{a}^T\boldsymbol{x}
  - \sum_j \log\!\left(1 + e^{\,b_j + [f_{\rm enc}(\boldsymbol{x})]_j}\right)$$

$$p(\boldsymbol{x}) \propto \exp(-F(\boldsymbol{x}))$$

This is **identical in form** to the BB-RBM free energy, with the CNN
pre-activations $f_{\rm enc}(\boldsymbol{x})$ playing the role of
$\boldsymbol{W}^T\boldsymbol{x}$.

### Conditional probabilities

**Hidden given visible** (lecture cell 211 generalised):

$$p(h_j=1\mid\boldsymbol{x}) = \sigma\!\left(b_j + [f_{\rm enc}(\boldsymbol{x})]_j\right)$$

**Visible given hidden** — the decoder gives pixel probabilities directly:

$$p(x_i=1\mid\boldsymbol{h}) = [f_{\rm dec}(\boldsymbol{h})]_i \in (0,1)$$

### Architecture overview

```
Encoder  f_enc : (B, 1, 28, 28) → (B, N_hidden)
  Conv(1→32,  3×3, pad=1) + BN + ReLU + MaxPool(2)  → (B, 32, 14, 14)
  Conv(32→64, 3×3, pad=1) + BN + ReLU + MaxPool(2)  → (B, 64,  7,  7)
  Flatten → Linear(3136 → N_hidden)                 → (B, N_hidden)

Decoder  f_dec : (B, N_hidden) → (B, 1, 28, 28)
  Linear(N_hidden → 3136) + ReLU                    → (B, 64, 7, 7)
  ConvTranspose(64→32, 4×4, stride=2) + BN + ReLU   → (B, 32, 14, 14)
  ConvTranspose(32→1,  4×4, stride=2) + Sigmoid      → (B,  1, 28, 28)
```

### Training

Contrastive divergence CD-$k$ via the free-energy loss:

$$\mathcal{L} = \mathbb{E}_{\boldsymbol{x}\sim p_{\rm data}}[F(\boldsymbol{x})]
              - \mathbb{E}_{\boldsymbol{x}^{(k)}\sim p_{\rm model}}[F(\boldsymbol{x}^{(k)})]$$

Minimising $\mathcal{L}$ lowers the free energy of real digits and raises
it for model-generated samples.

### Generation

After training, new digits are generated by running a long Gibbs chain
from random noise:

$$\boldsymbol{x}^{(0)} \sim \mathrm{Bernoulli}(0.5), \qquad
\boldsymbol{h}^{(t)} \sim p(\boldsymbol{h}\mid\boldsymbol{x}^{(t)}), \qquad
\boldsymbol{x}^{(t+1)} \sim p(\boldsymbol{x}\mid\boldsymbol{h}^{(t)})$$

---

## 2. Imports and Helpers

In [ ]:
%matplotlib inline
import math
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device         : {device}')
print(f'PyTorch version: {torch.__version__}')


def binarise(x, threshold=0.5):
    """
    Convert float pixel values in [0,1] to binary {0,1}.
    MNIST pixels > 0.5 become 1 (ink), others become 0 (background).
    This is required because the BB-RBM formalism assumes binary visible units.
    """
    return (x > threshold).float()


def show_grid(images, nrow=10, title='', figsize=(12, None), save_path=None):
    """Display a grid of 28×28 images."""
    if isinstance(images, torch.Tensor):
        images = images.detach().cpu().numpy()
    images = images.squeeze(1) if images.ndim == 4 else images
    N = len(images)
    nrows_fig = math.ceil(N / nrow)
    h = figsize[1] or nrows_fig * 1.6
    fig, axes = plt.subplots(nrows_fig, nrow, figsize=(figsize[0], h))
    for ax in axes.flatten(): ax.axis('off')
    for i, img in enumerate(images):
        axes.flatten()[i].imshow(img, cmap='gray_r', vmin=0, vmax=1,
                                  interpolation='nearest')
    if title:
        fig.suptitle(title, fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches='tight')
    plt.show()

## 3. CNN Encoder and Decoder

### Encoder $f_{\rm enc}$

Two convolutional blocks downsample the $28\times28$ image to $7\times7$
feature maps, which are then flattened and projected to the hidden dimension.
Batch normalisation after each convolution stabilises training.

The encoder outputs **pre-activations** (logits) — the sigmoid is applied
inside the free-energy formula so that autograd can correctly differentiate
$\log(1+e^{b_j+[f_{\rm enc}(\boldsymbol{x})]_j})$.

### Decoder $f_{\rm dec}$

Mirrors the encoder with transposed convolutions (sometimes called
*deconvolutions* or *fractionally-strided convolutions*).  The final
Sigmoid maps activations to pixel probabilities in $(0,1)$, making each
pixel a Bernoulli parameter.

In [ ]:
class CNNEncoder(nn.Module):
    """
    CNN encoder: x (B,1,28,28) → pre-activations (B, n_hidden).

    Replaces the linear map x → W^T x in the standard RBM.
    Uses spatial convolutions to exploit local pixel correlations.

    Output: pre-activation logits (NO sigmoid applied here).
    The sigmoid is applied inside free_energy / prob_h_given_x.
    """
    FLAT_DIM = 64 * 7 * 7   # 3136  — fixed for 28×28 input

    def __init__(self, n_hidden):
        super().__init__()
        self.conv = nn.Sequential(
            # Block 1: (B,1,28,28) → (B,32,14,14)
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            # Block 2: (B,32,14,14) → (B,64,7,7)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Linear(self.FLAT_DIM, n_hidden)

    def forward(self, x):
        B = x.size(0)
        z = self.conv(x.view(B, 1, 28, 28))   # (B, 64, 7, 7)
        return self.fc(z.view(B, -1))           # (B, n_hidden)  logits


class CNNDecoder(nn.Module):
    """
    CNN decoder: h (B, n_hidden) → pixel probabilities (B,1,28,28).

    Mirrors the encoder using transposed convolutions.
    The Sigmoid output gives p(x_i = 1 | h) for each pixel i.
    """
    FLAT_DIM = 64 * 7 * 7

    def __init__(self, n_hidden):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(n_hidden, self.FLAT_DIM),
            nn.ReLU(inplace=True),
        )
        self.deconv = nn.Sequential(
            # (B,64,7,7) → (B,32,14,14)
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            # (B,32,14,14) → (B,1,28,28)
            nn.ConvTranspose2d(32, 1,  kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),    # pixel probabilities in (0, 1)
        )

    def forward(self, h):
        B = h.size(0)
        z = self.fc(h).view(B, 64, 7, 7)   # reshape to spatial map
        return self.deconv(z)               # (B, 1, 28, 28)


# Quick shape tests
enc = CNNEncoder(n_hidden=256)
dec = CNNDecoder(n_hidden=256)
dummy_x = torch.zeros(4, 1, 28, 28)
dummy_h = torch.zeros(4, 256)
out_enc = enc(dummy_x)
out_dec = dec(dummy_h)
print(f'Encoder:  (4,1,28,28)  →  {tuple(out_enc.shape)}')
print(f'Decoder:  (4,256)      →  {tuple(out_dec.shape)}')
print(f'Decoder output range: [{out_dec.min():.3f}, {out_dec.max():.3f}]')

## 4. ConvRBM Class

The `ConvRBM` class wraps the encoder and decoder inside the RBM
probabilistic framework.  All methods mirror those of the `BinaryBinaryRBM_PT`
from the previous notebook, with the encoder/decoder replacing the weight matrix.

| BB-RBM | ConvRBM | Notes |
|:-------|:--------|:------|
| $p(h_j=1|\boldsymbol{x}) = \sigma(b_j + \boldsymbol{x}^T\boldsymbol{w}_{*j})$ | $\sigma(b_j + [f_{\rm enc}(\boldsymbol{x})]_j)$ | CNN replaces linear |
| $p(x_i=1|\boldsymbol{h}) = \sigma(a_i + \boldsymbol{w}_{i*}^T\boldsymbol{h})$ | $[f_{\rm dec}(\boldsymbol{h})]_i$ | ConvTranspose replaces linear |
| $F(\boldsymbol{x}) = -\boldsymbol{a}^T\boldsymbol{x} - \sum_j\log(1+e^{b_j+\boldsymbol{x}^T\boldsymbol{w}_{*j}})$ | same, with $f_{\rm enc}(\boldsymbol{x})$ instead of $\boldsymbol{W}^T\boldsymbol{x}$ | |
| $\mathcal{L} = F(\boldsymbol{x}_0) - F(\boldsymbol{x}_k)$ | same | CD-$k$ loss |

In [ ]:
class ConvRBM(nn.Module):
    """
    Convolutional Restricted Boltzmann Machine for 28×28 binary images.

    Energy model
    ------------
    Generalises the BB-RBM by replacing the linear encoder W^T x with
    a CNN  f_enc(x):

        E(x, h) = -a^T x  -  b^T h  -  h^T f_enc(x)

    Free energy (marginalise over h, lecture notes cell 205):

        F(x) = -a^T x  -  Σ_j log(1 + exp(b_j + [f_enc(x)]_j))
        p(x) ∝ exp(-F(x))

    Conditional probabilities
    -------------------------
        p(h_j=1 | x) = σ(b_j + [f_enc(x)]_j)    [lecture cell 211 form]
        p(x_i=1 | h) = [f_dec(h)]_i              [from ConvTranspose decoder]

    Gibbs sampling
    --------------
        h^(t) ~ Bernoulli( σ(b + f_enc(x^(t))) )
        x^(t+1) ~ Bernoulli( f_dec(h^(t)) )

    Training
    --------
        L = mean[ F(x_data) - F(x_k) ]   (CD-k contrastive loss)
    """

    def __init__(self, n_hidden=256, k=1):
        super().__init__()
        self.n_hidden  = n_hidden
        self.n_visible = 784       # 28 × 28
        self.k         = k

        self.encoder = CNNEncoder(n_hidden)
        self.decoder = CNNDecoder(n_hidden)

        # Biases (same role as in BB-RBM)
        self.a = nn.Parameter(torch.zeros(self.n_visible))  # visible biases
        self.b = nn.Parameter(torch.zeros(n_hidden))        # hidden  biases

        n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'ConvRBM: n_hidden={n_hidden}, k={k}, params={n_params:,}')

    # ── Conditional probabilities ──────────────────────────────────────

    def prob_h_given_x(self, x):
        """
        p(h_j=1 | x) = σ(b_j + [f_enc(x)]_j)   [lecture cell 211 form]
        x : (B,1,28,28)  →  (B, n_hidden)
        """
        return torch.sigmoid(self.encoder(x) + self.b)

    def prob_x_given_h(self, h):
        """
        p(x_i=1 | h) = [f_dec(h)]_i   — pixel-wise Bernoulli prob
        h : (B, n_hidden)  →  (B, 1, 28, 28)
        """
        return self.decoder(h)

    # ── Sampling ───────────────────────────────────────────────────────

    def sample_h(self, x):
        p = self.prob_h_given_x(x)
        return p, torch.bernoulli(p)

    def sample_x(self, h):
        p = self.prob_x_given_h(h)
        return p, torch.bernoulli(p)

    # ── Free energy ────────────────────────────────────────────────────

    def free_energy(self, x):
        """
        F(x) = -a^T x  -  Σ_j log(1 + exp(b_j + [f_enc(x)]_j))

        Derived from marginalising h out of the joint distribution
        (same derivation as lecture notes cell 205, with CNN pre-acts
        replacing the linear term x^T W).

        x : (B,1,28,28)  →  scalar per sample (B,)
        """
        # Visible bias term: -a^T x  (flatten x to (B,784))
        vis  = x.view(x.size(0), -1) @ self.a          # (B,)
        # Hidden term: -Σ_j log(1 + exp(b_j + [f_enc(x)]_j))
        pre  = self.encoder(x)                          # (B, n_hidden)
        hid  = torch.log1p(torch.exp(pre + self.b)).sum(1)  # (B,)
        return -(vis + hid)                             # (B,)

    # ── Gibbs chain ────────────────────────────────────────────────────

    def gibbs_k(self, x0):
        """k alternating Gibbs steps. Returns x_k (detached)."""
        x = x0
        for _ in range(self.k):
            _, h = self.sample_h(x)
            _, x = self.sample_x(h)
        return x.detach()

    def forward(self, x):
        """Return (x0, x_k) for the contrastive loss."""
        return x, self.gibbs_k(x)

    # ── Generation from scratch ────────────────────────────────────────

    @torch.no_grad()
    def generate(self, n_samples, n_gibbs=1000, init='noise'):
        """
        Generate new images by Gibbs sampling from random initialisation.

        Parameters
        ----------
        n_samples : number of images
        n_gibbs   : burn-in Gibbs steps
        init      : 'noise' (Bernoulli 0.5) or 'zeros'

        Returns
        -------
        prob_x : (n_samples, 1, 28, 28)  final pixel probabilities
        x_k    : (n_samples, 1, 28, 28)  final binary sample
        """
        self.eval()
        x = (torch.bernoulli(0.5 * torch.ones(n_samples, 1, 28, 28, device=device))
             if init == 'noise'
             else torch.zeros(n_samples, 1, 28, 28, device=device))

        for _ in range(n_gibbs):
            _, h = self.sample_h(x)
            _, x = self.sample_x(h)

        # Return both soft probabilities and hard binary sample
        prob_x = self.prob_x_given_h(torch.bernoulli(self.prob_h_given_x(x)))
        return prob_x, x


# Quick sanity check
model_test = ConvRBM(n_hidden=256, k=1)
xdummy = torch.zeros(2, 1, 28, 28)
F_dummy = model_test.free_energy(xdummy)
print(f'free_energy shape: {F_dummy.shape}  values: {F_dummy.detach().numpy().round(3)}')
_, xk = model_test(xdummy)
print(f'gibbs_k output shape: {xk.shape}  range: [{xk.min():.1f}, {xk.max():.1f}]')

## 5. MNIST Data Loading

MNIST images are originally 8-bit greyscale values in $[0,1]$ after
`ToTensor()`.  We binarise them at threshold $0.5$ to obtain the binary
visible units $\boldsymbol{x}\in\{0,1\}^{784}$ required by the BB-RBM
formulation.  Binarisation is done at training time (inside the loop)
so that the data loader remains standard.

In [ ]:
DATA_DIR   = './data'
BATCH_SIZE = 128

transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(DATA_DIR, train=True,
                                download=True, transform=transform)
test_dataset  = datasets.MNIST(DATA_DIR, train=False,
                                download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=0, pin_memory=True)

print(f'Train samples : {len(train_dataset):,}')
print(f'Test  samples : {len(test_dataset):,}')
print(f'Batch size    : {BATCH_SIZE}')

# Show a sample of binarised MNIST images
sample_imgs, sample_labels = next(iter(test_loader))
show_grid(binarise(sample_imgs[:40]), nrow=10,
          title='Real MNIST digits — binarised at threshold 0.5')

## 6. Training

**Hyperparameters:**

| Setting | Value | Rationale |
|:--------|:------|:----------|
| `n_hidden` | 256 | Enough capacity for MNIST without overfit |
| `k` (CD steps) | 1 | CD-1 is standard; larger $k$ improves quality but is slower |
| `epochs` | 20 | Sufficient for the loss to converge visibly |
| Optimiser | Adam | Adaptive LR handles different gradient scales in Conv vs Linear |
| LR | $3\times10^{-4}$ | Standard for Adam on MNIST-scale problems |
| Schedule | Cosine annealing | Smooth LR decay to 0 over the full run |
| Grad clip | norm $\le 1.0$ | Prevents occasional large spikes from CNN gradients |

Each batch step:
1. Binarise the input batch
2. Run $k=1$ Gibbs step to get $\boldsymbol{x}^{(k)}$ (negative phase)
3. Compute loss $\mathcal{L} = \mathrm{mean}[F(\boldsymbol{x}_0) - F(\boldsymbol{x}^{(k)})]$
4. Backprop through $F(\boldsymbol{x}_0)$ only ($\boldsymbol{x}^{(k)}$ is detached)
5. Clip gradients and update

In [ ]:
N_HIDDEN = 256
N_EPOCHS = 20
K_GIBBS  = 1
LR_INIT  = 3e-4

torch.manual_seed(42)
model = ConvRBM(n_hidden=N_HIDDEN, k=K_GIBBS).to(device)

optimizer = optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

print(f'Training ConvRBM on MNIST:')
print(f'  n_hidden={N_HIDDEN},  k={K_GIBBS},  epochs={N_EPOCHS},  lr={LR_INIT}')
print(f'  Loss = mean[ F(x_data) - F(x_k) ]  (CD-{K_GIBBS})\n')

train_losses = []
t_start = time.time()

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    ep_loss   = 0.0
    n_batches = 0
    t_ep      = time.time()

    for images, _ in train_loader:
        x  = binarise(images).to(device)          # (B,1,28,28) binary
        x0, x_k = model(x)                        # x0 = x, x_k = Gibbs sample

        # CD-k contrastive loss: F(data) - F(model)
        loss = (model.free_energy(x0) - model.free_energy(x_k)).mean()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        ep_loss   += loss.item()
        n_batches += 1

    scheduler.step()
    avg = ep_loss / n_batches
    train_losses.append(avg)
    print(f'  Epoch {epoch:3d}/{N_EPOCHS}  '
          f'loss = {avg:+.4f}  '
          f'lr = {scheduler.get_last_lr()[0]:.2e}  '
          f'({time.time()-t_ep:.1f}s)')

print(f'\nTotal training time: {time.time()-t_start:.1f}s')

In [ ]:
# Training loss curve
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(1, N_EPOCHS+1), train_losses, 'b-o', ms=5, lw=2)
ax.set(xlabel='Epoch',
       ylabel='CD loss   $F(\\mathbf{x}_{\\rm data}) - F(\\mathbf{x}_k)$',
       title='ConvRBM training on MNIST')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Generation — New Digit Images from Gibbs Sampling

Once trained, the model defines a probability distribution $p(\boldsymbol{x})$
over $28\times28$ binary images.  We sample from it by running a Markov
chain that converges to $p$:

$$\boldsymbol{x}^{(0)} \sim \mathrm{Bernoulli}(0.5) \quad (\text{random noise})$$

$$\boldsymbol{h}^{(t)} \sim \mathrm{Bernoulli}\!\left(\sigma(\boldsymbol{b}
  + f_{\rm enc}(\boldsymbol{x}^{(t)}))\right)$$

$$\boldsymbol{x}^{(t+1)} \sim \mathrm{Bernoulli}\!\left(f_{\rm dec}(\boldsymbol{h}^{(t)})\right)$$

After 1000 burn-in steps the chain has mixed and the final
$\boldsymbol{x}^{(1000)}$ is a sample from (approximately) $p$.
We also show the **soft pixel probabilities** $f_{\rm dec}(\boldsymbol{h})$
at the final step, which are smoother than the hard binary sample.

In [ ]:
model.eval()

# ── Reconstructions from test data ─────────────────────────────────────────
print('Reconstructions (one Gibbs step from test images):')
xb = binarise(sample_imgs[:40]).to(device)
with torch.no_grad():
    p_h   = model.prob_h_given_x(xb)
    h_s   = torch.bernoulli(p_h)
    recon = model.prob_x_given_h(h_s).cpu()

# Show original (top) and reconstruction (bottom) side by side
fig, axes = plt.subplots(2, 10, figsize=(14, 3.5))
for ax in axes.flatten(): ax.axis('off')
for i in range(10):
    axes[0, i].imshow(binarise(sample_imgs[i]).squeeze().numpy(),
                      cmap='gray_r', vmin=0, vmax=1)
    axes[1, i].imshow(recon[i].squeeze().numpy(),
                      cmap='gray_r', vmin=0, vmax=1)
axes[0, 0].set_ylabel('Input', fontsize=9)
axes[1, 0].set_ylabel('Recon', fontsize=9)
fig.suptitle('Top: binarised test images  |  Bottom: ConvRBM reconstructions',
             fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Generate new samples from random noise (1000 Gibbs steps) ────────────────
print('Generating 80 new digit images from random noise (1000 Gibbs steps)...')
gen_probs, gen_bin = model.generate(n_samples=80, n_gibbs=1000, init='noise')

show_grid(gen_probs.cpu(), nrow=10,
          title='Generated MNIST digits (ConvRBM, 1000 Gibbs steps from Bernoulli noise)')

In [ ]:
# ── Generated samples from all-zero initialisation ───────────────────────────
print('Generating from all-zero initialisation...')
gen_zeros, _ = model.generate(n_samples=40, n_gibbs=1000, init='zeros')

show_grid(gen_zeros.cpu(), nrow=10,
          title='Generated from all-zero start (1000 Gibbs steps)')

In [ ]:
# ── Gibbs chain evolution ─────────────────────────────────────────────────────
print('Tracking one Gibbs chain over 800 steps from random noise...')
snap_steps = [0, 5, 20, 50, 100, 200, 400, 800]
snapshots  = []

with torch.no_grad():
    x_chain = torch.bernoulli(
        0.5 * torch.ones(1, 1, 28, 28, device=device))
    if 0 in snap_steps:
        snapshots.append((0, x_chain.cpu()))
    for t in range(1, 801):
        _, hc = model.sample_h(x_chain)
        _, x_chain = model.sample_x(hc)
        if t in snap_steps:
            snapshots.append((t, x_chain.cpu()))

fig, axes = plt.subplots(1, len(snapshots), figsize=(len(snapshots)*1.8, 2.3))
for ax, (t, img) in zip(axes, snapshots):
    ax.imshow(img.squeeze().numpy(), cmap='gray_r', vmin=0, vmax=1)
    ax.set_title(f't={t}', fontsize=9)
    ax.axis('off')
fig.suptitle('Gibbs chain evolution  (random start → converged digit)',
             fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Free energy of real digits vs generated samples ──────────────────────────
print('Free energy comparison: real digits vs generated samples')
with torch.no_grad():
    F_real = model.free_energy(binarise(sample_imgs[:200]).to(device)).cpu()
    F_gen  = model.free_energy(gen_bin[:80].to(device)).cpu()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(F_real.numpy(), bins=30, alpha=0.7, color='steelblue',
        label=f'Real MNIST  (mean={F_real.mean():.1f})')
ax.hist(F_gen.numpy(),  bins=30, alpha=0.7, color='firebrick',
        label=f'Generated   (mean={F_gen.mean():.1f})')
ax.set(xlabel='Free energy $F(\\mathbf{x})$',
       ylabel='Count',
       title='Free energy distribution\n'
             '$p(\\mathbf{x}) \\propto \\exp(-F(\\mathbf{x}))$ '
             '— lower $F$ = higher probability')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('\nObservation: generated samples and real digits should have')
print('overlapping free energy distributions when the model is well-trained.')

## 8. Reconstruction Quality

We measure the **per-pixel MSE** between binarised test images and their
one-step reconstructions:

$$\text{MSE} = \frac{1}{N}\sum_n\frac{1}{784}\sum_i
\left(x_{ni} - [f_{\rm dec}(\boldsymbol{h}^{(1)}_n)]_i\right)^2$$

A lower MSE means the model has learned to encode and decode digit structure
faithfully.  We also break it down by digit class to check whether some
digits are harder to reconstruct than others.

In [ ]:
model.eval()
total_mse = 0.0; n_test = 0
with torch.no_grad():
    for images, _ in test_loader:
        xb    = binarise(images).to(device)
        h_s   = torch.bernoulli(model.prob_h_given_x(xb))
        recon = model.prob_x_given_h(h_s)
        total_mse += ((xb - recon)**2).mean(dim=[1,2,3]).sum().item()
        n_test    += len(xb)

avg_mse = total_mse / n_test
print(f'Test-set reconstruction MSE: {avg_mse:.5f}  '
      f'(per pixel, {n_test:,} images)\n')

# Per-class breakdown
class_mse = {d: 0.0 for d in range(10)}
class_cnt = {d: 0   for d in range(10)}
with torch.no_grad():
    for images, labels in test_loader:
        xb    = binarise(images).to(device)
        h_s   = torch.bernoulli(model.prob_h_given_x(xb))
        recon = model.prob_x_given_h(h_s)
        mse_pp = ((xb - recon)**2).mean(dim=[1,2,3])
        for d in range(10):
            mask = (labels == d)
            if mask.any():
                class_mse[d] += mse_pp[mask].sum().item()
                class_cnt[d] += mask.sum().item()

print(f'  {"Digit":>6s}  {"MSE":>8s}  {"N":>6s}')
print('  ' + '-'*24)
for d in range(10):
    m = class_mse[d] / max(class_cnt[d], 1)
    print(f'  {d:>6d}  {m:>8.5f}  {class_cnt[d]:>6d}')

# Per-class MSE bar chart
fig, ax = plt.subplots(figsize=(8, 3.5))
mses = [class_mse[d]/max(class_cnt[d],1) for d in range(10)]
ax.bar(range(10), mses, color='steelblue', alpha=0.8)
ax.axhline(avg_mse, color='firebrick', ls='--', lw=1.5,
           label=f'Overall mean MSE = {avg_mse:.4f}')
ax.set(xlabel='Digit class', ylabel='Reconstruction MSE',
       title='ConvRBM reconstruction MSE per digit class')
ax.set_xticks(range(10))
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 9. Summary

### What was built

A **Convolutional RBM** that uses a CNN as the energy function, trained on
MNIST with CD-1 to learn the distribution of handwritten digits and generate
new ones by Gibbs sampling.

### Relationship to the lecture-note equations

| Equation | Lecture cell | ConvRBM implementation |
|:---------|:------------:|:-----------------------|
| $E(x,h) = -a^Tx - b^Th - x^TWh$ | 199 | `E = -a·x - b·h - h·f_enc(x)` |
| $p(h_j=1|x) = \sigma(b_j+x^Tw_{*j})$ | 211 | `torch.sigmoid(encoder(x) + b)` |
| $p(x_i=1|h) = \sigma(a_i+w_{i*}^Th)$ | 217 | `decoder(h)` (ConvTranspose + Sigmoid) |
| $F(x) = -a^Tx - \sum_j\log(1+e^{b_j+x^Tw_{*j}})$ | 205 | Same formula, CNN pre-acts replace linear |
| $\mathcal{L} = F(x_0) - F(x_k)$ | CD loss | `(free_energy(x0) - free_energy(xk)).mean()` |

### Why convolutional?

The flat BB-RBM weight matrix $\boldsymbol{W}\in\mathbb{R}^{784\times N}$
treats every pixel independently.  The CNN encoder shares weights across
spatial positions via convolution filters, letting the model detect local
features (edges, curves) that compose into digit shapes — a much stronger
inductive bias for image data.  This reduces the effective parameter count
for the encoder while improving the quality of generated samples.

### Extensions

- **Deeper networks**: add more conv blocks, or use residual connections
- **Larger $k$**: CD-$k$ with $k>1$ or Persistent CD (PCD) gives a better
  approximation of the negative-phase gradient
- **Continuous units**: replace Bernoulli visible units with Gaussian
  (Gaussian-binary RBM) for un-binarised images
- **Conditional generation**: add a class label as an extra input to the
  encoder to generate digit images of a specific class